# H3N2 HA Mutation Landscape EDA

Exploratory analysis of mutation landscapes built from NCBI sequences using gget virus.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load dataset
data_dir = Path('../data/landscapes')
with open(data_dir / 'h3n2_ha_landscapes.json') as f:
    dataset = json.load(f)

print(f"Loaded dataset with {len(dataset['landscapes'])} landscapes")
print(f"Years: {sorted(dataset['landscapes'].keys())[:5]}...{sorted(dataset['landscapes'].keys())[-5:]}")

## 1. Dataset Overview

In [ ]:
# Summary statistics
years = sorted([int(y) for y in dataset['landscapes'].keys()])
n_landscapes = len(dataset['landscapes'])

print(f"Total landscapes: {n_landscapes}")
print(f"Year range: {min(years)} - {max(years)}")
print(f"Protein: {dataset['metadata']['protein']}")
print(f"Total sequences: {dataset['metadata']['n_sequences']}")

# Sequences per year
seqs_per_year = [dataset['statistics'][str(y)]['n_sequences'] for y in years]
print(f"\nSequences per year: min={min(seqs_per_year)}, max={max(seqs_per_year)}, mean={np.mean(seqs_per_year):.1f}")

## 2. Landscape Statistics

In [ ]:
# Extract statistics for all years
entropy_means = []
conservation_means = []
n_conserved = []
n_diverse = []

for year in years:
    stats = dataset['statistics'][str(year)]
    entropy_means.append(stats['entropy_mean'])
    conservation_means.append(stats['conservation_mean'])
    n_conserved.append(stats.get('n_highly_conserved', 0))
    n_diverse.append(stats.get('n_high_diversity', 0))

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(years, entropy_means, 'o-', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Mean Shannon Entropy')
axes[0, 0].set_title('Sequence Diversity Over Time')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(years, conservation_means, 'o-', linewidth=2, markersize=6, color='orange')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Mean Conservation Score')
axes[0, 1].set_title('Sequence Conservation Over Time')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(years, n_conserved, 'o-', linewidth=2, markersize=6, color='green')
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('N Highly Conserved Positions')
axes[1, 0].set_title('Conserved Positions (>80% WT) Over Time')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(years, n_diverse, 'o-', linewidth=2, markersize=6, color='red')
axes[1, 1].set_xlabel('Year')
axes[1, 1].set_ylabel('N High Diversity Positions')
axes[1, 1].set_title('High Diversity Positions (H>3 bits) Over Time')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Entropy range: {min(entropy_means):.3f} - {max(entropy_means):.3f}")
print(f"Conservation range: {min(conservation_means):.3f} - {max(conservation_means):.3f}")

## 3. Temporal Dynamics

In [ ]:
# Year-to-year landscape changes
if dataset.get('temporal_dynamics'):
    temporal = dataset['temporal_dynamics']
    
    year_pairs = [(t['year_from'], t['year_to']) for t in temporal]
    distances = [t['js_distance'] for t in temporal]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    x = range(len(year_pairs))
    ax.bar(x, distances, color='steelblue', alpha=0.7)
    ax.set_xlabel('Year Transition')
    ax.set_ylabel('Jensen-Shannon Distance')
    ax.set_title('Year-to-Year Landscape Changes')
    ax.set_xticks(x[::2])  # Every other label for readability
    ax.set_xticklabels([f"{y[0]}-{y[1]}" for y in year_pairs[::2]], rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    print(f"Mean JS distance: {np.mean(distances):.4f}")
    print(f"Max JS distance: {np.max(distances):.4f} (year {year_pairs[np.argmax(distances)]})")
    print(f"Min JS distance: {np.min(distances):.4f} (year {year_pairs[np.argmin(distances)]})")

## 4. Sample Landscape Matrix

In [ ]:
# Visualize a single year's landscape
year_to_visualize = str(years[-1])  # Most recent year
landscape = np.array(dataset['landscapes'][year_to_visualize])

print(f"Landscape shape: {landscape.shape} (positions × amino acids)")
print(f"Example position 100:")
print(f"  {landscape[100]}")

# Heatmap of first 50 positions × 20 amino acids
aa_labels = list('ACDEFGHIKLMNPQRSTVWY')
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(landscape[:100].T, cmap='Blues', aspect='auto')
ax.set_xlabel('Position')
ax.set_ylabel('Amino Acid')
ax.set_title(f'H3N2 HA Mutation Landscape - Year {year_to_visualize} (positions 1-100)')
ax.set_yticks(range(20))
ax.set_yticklabels(aa_labels)
plt.colorbar(im, ax=ax, label='Frequency')
plt.tight_layout()
plt.show()

## 5. Prepare for Flow Matching

Structure data for training evolutionary trajectory model

In [ ]:
# Build training data: (sequence history) → (future landscape)
# For each year t, collect sequences up to t and target landscape at t+1

training_windows = []

for i, year_t in enumerate(years[:-1]):
    year_t_next = years[i+1]
    
    # Input: all landscapes up to year t
    input_landscapes = np.array([dataset['landscapes'][str(y)] for y in years[:i+1]])
    
    # Target: landscape at year t+1
    target_landscape = np.array(dataset['landscapes'][str(year_t_next)])
    
    # Metadata
    input_n_seqs = sum([dataset['statistics'][str(y)]['n_sequences'] for y in years[:i+1]])
    target_n_seqs = dataset['statistics'][str(year_t_next)]['n_sequences']
    
    training_windows.append({
        'year_t': year_t,
        'year_t_next': year_t_next,
        'n_input_years': i+1,
        'input_n_sequences': input_n_seqs,
        'target_n_sequences': target_n_seqs,
    })

print(f"Created {len(training_windows)} training windows")
print(f"\nExample training window:")
tw = training_windows[10]
print(f"  Predict {tw['year_t_next']} from sequences up to {tw['year_t']}")
print(f"  Input: {tw['n_input_years']} years, {tw['input_n_sequences']} sequences")
print(f"  Target: {tw['target_n_sequences']} sequences")